# Slot-Builder LoRA Trainer
## Notebook version — no command line

This notebook trains the slot-builder adapter from the v73b corpus.

Current trunk:

$$
Q_{\text{real}} \rightarrow C_{\text{patched}} \rightarrow \Delta W_{\text{slot}}
$$

Expected folder layout:

```text
Downloads/
  train_slot_builder_lora_notebook.ipynb
  v73b_outputs_local_root_prompt_recovered_slot_corpus/
    slot_sft_messages.jsonl
```

Output:

```text
Downloads/
  slot_builder_lora_v1/
```

Training target:

$$
Q \rightarrow C
$$

not:

$$
Q \rightarrow \text{answer}
$$


In [9]:
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path

ROOT = Path.cwd()
TRAIN_FILE = ROOT / "v73b_outputs_local_root_prompt_recovered_slot_corpus" / "slot_sft_messages.jsonl"
OUTPUT_DIR = ROOT / "slot_builder_lora_v1"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# If the model is already cached locally, this avoids internet lookup.
# Set False only if you need Hugging Face to download the model.
LOCAL_FILES_ONLY = True

# RTX 4060-safe defaults.
MAX_SEQ_LENGTH = 1536
EPOCHS = 20
LEARNING_RATE = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 8

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

EVAL_FRACTION = 0.15
SEED = 42

# Use 4-bit only if bitsandbytes works in your environment.
# For Qwen 1.5B on a 4060, normal fp16 LoRA usually fits.
USE_4BIT = False

RUN_SMOKE_TEST = True

print("Notebook root:", ROOT)
print("Train file:", TRAIN_FILE)
print("Output dir:", OUTPUT_DIR)
print("Train file exists:", TRAIN_FILE.exists())


Notebook root: D:\@User Data\Downloads
Train file: D:\@User Data\Downloads\v73b_outputs_local_root_prompt_recovered_slot_corpus\slot_sft_messages.jsonl
Output dir: D:\@User Data\Downloads\slot_builder_lora_v1
Train file exists: True


## Dependency check

Run the next cell. It imports the needed packages.

If an import fails, set `INSTALL_MISSING = True` and rerun the cell.


In [10]:
# ============================================================
# IMPORTS / OPTIONAL INSTALL
# ============================================================
INSTALL_MISSING = False

if INSTALL_MISSING:
    import sys, subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-U",
        "transformers", "peft", "accelerate", "sentencepiece", "datasets"
    ])

import json
import random
from typing import Any, Dict, List

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

set_seed(SEED)

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))


torch: 2.11.0+cu126
cuda: True
gpu: NVIDIA GeForce RTX 4060
vram GB: 8.0


In [11]:
# ============================================================
# LOAD v73b CORPUS
# ============================================================
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

if not TRAIN_FILE.exists():
    raise FileNotFoundError(
        f"Missing training file:\n{TRAIN_FILE}\n\n"
        "Keep this notebook in Downloads beside the v73b output folder."
    )

rows = read_jsonl(TRAIN_FILE)
print("rows:", len(rows))
print("first row keys:", rows[0].keys())
print("first base_id:", rows[0].get("base_id"))
print("target source:", rows[0].get("target_source"))

# Show first training exchange.
rows[0]["messages"]


rows: 24
first row keys: dict_keys(['messages', 'base_id', 'target_source', 'prompt_source'])
first base_id: adv_api_01
target source: patched


[{'role': 'system',
  'content': 'You are the Nexus Slot Constructor.\n\nYour job is to generate the missing-shape contract before answer selection.\nDo not answer the task.\nDo not mention answer choices.\nReturn strict JSON only.\n\nThe contract must contain:\nfamily_class\ndomain_carrier\nforbidden_neighbor_carrier\nboundary_conditions\npreserved_function\nfailure_modes\nwitness_readout\nresidue\n\nUse operational fit, not labels.\n'},
 {'role': 'user',
  'content': 'Prompt:\nAn API call exposes one method while hiding authentication, routing, validation, persistence, retries, and errors. What is the operational event?\n\nGenerate the missing-shape contract.\n\nChecklist:\n1. Need: occupy the inverse cavity.\n2. Function: preserve or redirect the required operation.\n3. Boundary: respect constraints.\n4. Trap: reject noun/surface-label confusion.\n5. Collapse: produce one executable witness/readout.\n\nReturn JSON only.'},
 {'role': 'assistant',
  'content': '{\n  "family_class": "m

In [12]:
# ============================================================
# DATASET
# ============================================================
def render_chat(tokenizer, messages: List[Dict[str, str]], add_generation_prompt: bool = False) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

    # Fallback for non-chat tokenizers.
    out = []
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")
        out.append(f"{role.upper()}:\n{content}")
    if add_generation_prompt:
        out.append("ASSISTANT:\n")
    return "\n\n".join(out)

class ChatSFTDataset(Dataset):
    # Causal-LM SFT with prompt masking.
    # Loss is applied only to the assistant contract JSON.

    def __init__(self, rows: List[Dict[str, Any]], tokenizer, max_seq_length: int):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        row = self.rows[idx]
        messages = row["messages"]

        if len(messages) < 3 or messages[-1].get("role") != "assistant":
            raise ValueError(f"Row {idx} must end with an assistant message.")

        prompt_messages = messages[:-1]
        assistant_message = messages[-1]

        prompt_text = render_chat(self.tokenizer, prompt_messages, add_generation_prompt=True)
        full_text = prompt_text + assistant_message["content"]
        if self.tokenizer.eos_token:
            full_text += self.tokenizer.eos_token

        prompt_ids = self.tokenizer(
            prompt_text,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_seq_length,
        )["input_ids"]

        full = self.tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_seq_length,
        )

        input_ids = full["input_ids"]
        attention_mask = full["attention_mask"]

        labels = input_ids.copy()
        prompt_len = min(len(prompt_ids), len(labels))
        labels[:prompt_len] = [-100] * prompt_len

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

class PadCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    def __call__(self, batch):
        max_len = max(x["input_ids"].shape[0] for x in batch)

        input_ids = []
        attention_mask = []
        labels = []

        for x in batch:
            n = x["input_ids"].shape[0]
            pad = max_len - n

            input_ids.append(torch.cat([x["input_ids"], torch.full((pad,), self.pad_id, dtype=torch.long)]))
            attention_mask.append(torch.cat([x["attention_mask"], torch.zeros(pad, dtype=torch.long)]))
            labels.append(torch.cat([x["labels"], torch.full((pad,), -100, dtype=torch.long)]))

        return {
            "input_ids": torch.stack(input_ids),
            "attention_mask": torch.stack(attention_mask),
            "labels": torch.stack(labels),
        }

def split_rows(rows, seed=42, eval_fraction=0.15):
    rng = random.Random(seed)
    rows = list(rows)
    rng.shuffle(rows)

    if len(rows) < 6 or eval_fraction <= 0:
        return rows, []

    n_eval = max(1, int(round(len(rows) * eval_fraction)))
    eval_rows = rows[:n_eval]
    train_rows = rows[n_eval:]
    return train_rows, eval_rows

train_rows, eval_rows = split_rows(rows, SEED, EVAL_FRACTION)
print("train rows:", len(train_rows))
print("eval rows:", len(eval_rows))


train rows: 20
eval rows: 4


In [13]:
# ============================================================
# LOAD TOKENIZER + BASE MODEL
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    local_files_only=LOCAL_FILES_ONLY,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    "local_files_only": LOCAL_FILES_ONLY,
    "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
}

if USE_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs["device_map"] = "auto"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)

if not USE_4BIT and torch.cuda.is_available():
    model = model.to("cuda")

model.config.use_cache = False

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

print("loaded:", MODEL_NAME)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

loaded: Qwen/Qwen2.5-1.5B-Instruct


In [14]:
# ============================================================
# ATTACH LoRA
# ============================================================
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [15]:
# ============================================================
# TRAIN
# ============================================================
train_ds = ChatSFTDataset(train_rows, tokenizer, MAX_SEQ_LENGTH)
eval_ds = ChatSFTDataset(eval_rows, tokenizer, MAX_SEQ_LENGTH) if eval_rows else None

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    logging_steps=1,
    save_strategy="epoch",
    eval_strategy="epoch" if eval_ds is not None else "no",
    fp16=torch.cuda.is_available(),
    bf16=False,
    optim="adamw_torch",
    report_to=[],
    remove_unused_columns=False,
    gradient_checkpointing=True,
    save_total_limit=3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=PadCollator(tokenizer),
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,1.536407,1.665525
2,1.069236,1.280434
3,0.754551,0.967638
4,0.593535,0.804049
5,0.438657,0.750149
6,0.378772,0.688908
7,0.242174,0.715972
8,0.168018,0.734232
9,0.118855,0.812321
10,0.059195,0.891635


TrainOutput(global_step=60, training_loss=0.3085739608698835, metrics={'train_runtime': 212.9918, 'train_samples_per_second': 1.878, 'train_steps_per_second': 0.282, 'total_flos': 1316314605465600.0, 'train_loss': 0.3085739608698835, 'epoch': 20.0})

In [16]:
# ============================================================
# SAVE ADAPTER + CONFIG
# ============================================================
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

config = {
    "model_name": MODEL_NAME,
    "train_file": str(TRAIN_FILE),
    "output_dir": str(OUTPUT_DIR),
    "rows": len(rows),
    "train_rows": len(train_rows),
    "eval_rows": len(eval_rows),
    "max_seq_length": MAX_SEQ_LENGTH,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "use_4bit": USE_4BIT,
}

(OUTPUT_DIR / "slot_builder_training_config.json").write_text(
    json.dumps(config, indent=2),
    encoding="utf-8"
)

print("Adapter saved to:", OUTPUT_DIR)
print("Files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)


Adapter saved to: D:\@User Data\Downloads\slot_builder_lora_v1
Files:
 - adapter_config.json
 - adapter_model.safetensors
 - chat_template.jinja
 - checkpoint-54
 - checkpoint-57
 - checkpoint-60
 - README.md
 - slot_builder_training_config.json
 - tokenizer.json
 - tokenizer_config.json


In [17]:
# ============================================================
# SMOKE TEST: generate a few contracts
# ============================================================
def smoke_test(model, tokenizer, rows, output_dir: Path, max_new_tokens: int = 450):
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    out_path = output_dir / "smoke_test_generations.jsonl"

    sample_rows = rows[: min(5, len(rows))]
    generations = []

    with out_path.open("w", encoding="utf-8") as f:
        for row in sample_rows:
            messages = row["messages"][:-1]
            expected = row["messages"][-1]["content"]

            prompt_text = render_chat(tokenizer, messages, add_generation_prompt=True)
            inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

            with torch.no_grad():
                generated = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )

            new_tokens = generated[0, inputs["input_ids"].shape[1]:]
            pred = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

            rec = {
                "base_id": row.get("base_id"),
                "target_source": row.get("target_source"),
                "expected_contract": expected,
                "generated_contract": pred,
            }
            generations.append(rec)
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print("Smoke test written:", out_path)
    return generations

if RUN_SMOKE_TEST:
    generations = smoke_test(model, tokenizer, rows, OUTPUT_DIR)
    generations[0]
else:
    print("Smoke test skipped.")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Smoke test written: D:\@User Data\Downloads\slot_builder_lora_v1\smoke_test_generations.jsonl


## Readout

If training finishes, the trunk artifact is:

```text
slot_builder_lora_v1/
```

That folder is the slot-builder adapter:

$$
\Delta W_{\text{slot}}
$$

Next use:

```text
Base model + slot_builder_lora_v1
  ↓
generate missing-shape contracts
  ↓
operational critic
  ↓
KRRB resolver
```

No new branch. This is the current trunk.


In [18]:
from pathlib import Path
import json

SMOKE_FILE = Path.cwd() / "slot_builder_lora_v1" / "smoke_test_generations.jsonl"

rows = []
with SMOKE_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

print("smoke rows:", len(rows))

for i, r in enumerate(rows):
    print("=" * 100)
    print("CASE", i, r.get("base_id"))
    print("\nGENERATED:")
    print(r["generated_contract"])
    print("\nEXPECTED:")
    print(r["expected_contract"])

smoke rows: 5
CASE 0 adv_api_01

GENERATED:
{
  "family_class": "missing-shape",
  "domain_carrier": [
    "api",
    "call",
    "exposes",
    "one",
    "method",
    "hiding",
    "authentication",
    "routing",
    "validation",
    "persistence",
    "retries",
    "errors"
  ],
  "forbidden_neighbor_carrier": [
    "authentication",
    "routing",
    "validation",
    "persistence",
    "retries",
    "errors"
  ],
  "boundary_conditions": [
    "{'valid': 'method exposed', 'invalid': ['hidden authentication', 'hides routing', 'hides validation', 'hides persistence', 'hides retries', 'hides errors']}"
  ],
  "preserved_function": "operational event",
  "failure_modes": [
    "incorrect method exposure",
    "unauthorized access",
    "incomplete data handling",
    "improper error management",
    "lack of session tracking",
    "inadequate logging"
  ],
  "witness_readout": "an API call exposing one method while hiding authentication, routing, validation, persistence, retries